In [1]:
import sys, os
dir = os.getcwd()

ext = ['', '/..', '/../src/models', '/../src/nlp', '/../src/synth']
sys.path += [dir + i for i in ext]

In [2]:
IMGBB_API_KEY = '8ae6cb592a734c0c3593f1934b9d90a9'
OPENAI_API_KEY = 'YOUR_OPENAI_KEY'

In [3]:
import base64
import requests

def upload_im(im, key):
    url = "https://api.imgbb.com/1/upload"
    payload = {
        'key': key,
        'image': base64.b64encode(im)
    }
    response = requests.post(url, data=payload)
    if response.status_code == 200:
        return response.json()['data']['url']
    else:
        raise Exception(f'[{response.status_code}] Failed to upload image.')

In [4]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

class Chat2:

    class Entry:
        def __init__(self, role: str, text: str = "", im = None, im_url: str = ''):
            self.role = role
            self.text = text
            if im_url:
                self.im_url = im_url
            elif im:
                self.im_url = upload_im(im, IMGBB_API_KEY)
            else:
                self.im_url = ''

        def to_dict(self) -> dict:
            content = []

            if self.text:
                content.append({
                    'type': 'text',
                    'text': self.text
                })

            if self.im_url:
                content.append({
                    'type': 'image_url',
                    'image_url': {
                        'url': self.im_url
                    }
                })

            obj = {
                'role': self.role,
                'content': content
            }

            return obj

    def __init__(self, client, model='gpt-4o-mini'):
        self.model = model
        self.client = client

    def __call__(self, input: list[Entry] = [], json: bool = False):
        chat = self.client.chat.completions.create(
            model=self.model,
            messages=[entry.to_dict() for entry in input],
            response_format={'type': 'json_object' if json else 'text'}
        )
        return chat.choices[0].message.content

In [19]:
import os
import cv2
import base64
import tiktoken

def extract_frames_from_video(video_path, output_dir=None, time_step=0.2, scale_factor=1.0):

    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Video file not found: {video_path}")
        
    if output_dir is None:
        video_dir = os.path.dirname(video_path)
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        output_dir = os.path.join(video_dir, f"{video_name}_frames")
    
    os.makedirs(output_dir, exist_ok=True)
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Error opening video file: {video_path}")
    
    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS)

    frame_interval = int(fps * time_step)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Get original dimensions
    original_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    original_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Calculate new dimensions
    new_width = int(original_width * scale_factor)
    new_height = int(original_height * scale_factor)
    
    frame_paths = []
    frame_count = 0
    current_frame = 0
    total_tokens = 0
    
    # Initialize tokenizer
    encoding = tiktoken.get_encoding("cl100k_base")  # GPT-4 encoding
    
    while current_frame < total_frames:
        cap.set(cv2.CAP_PROP_POS_FRAMES, current_frame)
        ret, frame = cap.read()
        
        if not ret:
            break
            
        # Resize frame if scale_factor is not 1.0
        if scale_factor != 1.0:
            frame = cv2.resize(frame, (new_width, new_height), interpolation=cv2.INTER_AREA)
        
        frame_path = os.path.join(output_dir, f"frame_{frame_count:04d}.jpg")
        cv2.imwrite(frame_path, frame)
        frame_paths.append(frame_path)
        
        # Calculate tokens for this frame
        # Convert frame to base64 string (similar to how it would be sent to API)
        _, buffer = cv2.imencode('.jpg', frame)
        base64_string = base64.b64encode(buffer).decode('utf-8')
        frame_tokens = len(encoding.encode(base64_string))
        total_tokens += frame_tokens
        
        frame_count += 1
        current_frame += frame_interval
    
    cap.release()
    
    print(f"Successfully extracted {frame_count} frames to {output_dir}")
    print(f"Time interval between frames: {time_step} seconds")
    print(f"Frame dimensions: {new_width}x{new_height} (scale factor: {scale_factor})")
    print(f"Total estimated tokens for all frames: {total_tokens}")
    print(f"Average tokens per frame: {total_tokens // frame_count if frame_count > 0 else 0}")
    
    return frame_paths, total_tokens

frames, total_tokens = extract_frames_from_video(
    '/Users/jdiazchao/Documents/narrated_demo/narrated_demo/examples/transcript_7/data/demonstration0/videos/participant0_demo0_segment0.mp4',
    time_step=2.0,
    scale_factor = 0.4
)

Successfully extracted 20 frames to /Users/jdiazchao/Documents/narrated_demo/narrated_demo/examples/transcript_7/data/demonstration0/videos/participant0_demo0_segment0_frames
Time interval between frames: 2.0 seconds
Frame dimensions: 432x768 (scale factor: 0.4)
Total estimated tokens for all frames: 913366
Average tokens per frame: 45668


In [6]:
def get_im(dir):
    with open(dir, "rb") as im: 
        return im.read()

### Import **API**

In [7]:
from api.football import *

### Import **Scene**

In [8]:
import string
alph = list(string.ascii_lowercase)

In [9]:
# Automatic

FOLDER = os.path.join('transcript_7/data')
demos = [i for i in os.listdir(FOLDER) if i.startswith('demonstration')]
demos.sort()

print('Found demos:', demos)

Found demos: ['demonstration0', 'demonstration1']


In [10]:
import string
from scene import Scene
from api.objects.registry import REGISTRY # TODO: Rename to ObjectsAPI

map = {}

for i, d in enumerate(demos):

    map[str(i + 1)] = {}

    folder = os.path.join(FOLDER, d, 'json_segments')
    diri = [i for i in os.listdir(folder) if i.endswith('.json')]
    diri.sort()

    for j, f in enumerate(diri):
        file = os.path.join(folder, f)
        with open(file) as f:
            data = json.load(f)

        map[str(i + 1)][alph[j]] = Scene.from_dict(data['scene'], REGISTRY)

In [11]:
narrations = '\n'.join([f"Demo {j + 1}: " + ' '.join([f"({alph[idx]}) {s.language}" for idx, s in enumerate(i.values())]) for j, i in enumerate(map.values())])
print(narrations)

Demo 1: (a) First,get possession of the ball. Then,pass the [The Coach passed the ball to teammate in the scene.] ball to your teammate [The teammate passed the ball to Coach in the scene.].Once your teammate returns the pass,as the teammate breaches or passes through the enemy line, pass the ball to your teammate.
Demo 2: (a) First,you need to take possession of the ball . [The Coach passed the ball to teammate in the scene.] Then pass the ball to your teammate . [The teammate passed the ball to Coach in the scene.] Once the teammate returns a ball to you , [The Coach passed the ball to teammate in the scene.] as a teammate goes past the enemy line,pass the ball to your teammate.


In [12]:
inference_instructions = """
The user inputs a series of top-down keyframes (in chronological order) from a video where a soccer coach instructs on how to play under a particular scenario as if they were a player.
You're tasked to reason about the keyframes in the context of the coach's explanation and demonstrations and build a series of tasks for the player.
We would like to learn such behaviour from the coach for another player replacing them so tasks should be defined for the coach or hypothetical player replacing the coach.

Tasks are defined in the context of a finite state machine were each task denotes a node with a termination condition, i.e., the condition under which the task will be marked as terminated so the state can change,
and edges are preconditions, i.e., the condition under which a following task should be triggered and the state changes from the previous task.

You response return must be a json with format 

{
    'tasks': [
        {
            'objective': str,
            'control': str,
            'termination': str,
            'condition': str,
        }
    ]
}

Tasks should be in order as they should be performed. Note that for every tasks there is: (1) an objective, the task specification; (2) the control, how to do perform the task; (3) the termination, the conditions under which such task should be interrupted; (4) and a condition, the conditions under which such task gets triggered.
Note that (2), (3) and (4) could be empty or None if not specified by the coach. If no condition or termination conditions was specified or is implied maybe by domain logic, set to empty or None, the assumption is that task will happen sequentially then.

Note that careful frame selection is important for accurate behavior learning (e.g. when we learn when to make a pass we want to learn from the frame where the pass was made and both players are at the expected locations).
The subtasks should be ordered chronologically too.
"""

entries = [
    Chat2.Entry(role='system', text=inference_instructions),
    Chat2.Entry(role='user', text=narrations)
] + [
    Chat2.Entry(role='user', text=f'Frame index: {idx}', im=get_im(im_dir)) for idx, im_dir in enumerate(frames)
]

chat = Chat2(client, model='gpt-4o')
response = json.loads(chat(entries, json=True))
print('Inference done.')

Inference done.


In [13]:
from task import Task

tasks = []
for t in response['tasks']:
    tasks.append(Task(len(tasks), t['objective'], t['control'], t['termination'], t['condition'], [str(idx + 1) for idx, _ in enumerate(demos)]))
print('\n\n'.join([str(t) + f'\nSources: {str(t.sources)}' for t in tasks]))

Task ID: 0
Objective (what): Get possession of the ball.
Control (how): Move towards the ball and control it.
Termination (until): Ball is under control.
Condition (when): 
Sources: ['1', '2']

Task ID: 1
Objective (what): Pass the ball to a teammate.
Control (how): Identify the teammate's position and pass.
Termination (until): Ball reaches the teammate.
Condition (when): 
Sources: ['1', '2']

Task ID: 2
Objective (what): Receive the ball from the teammate.
Control (how): Position yourself to receive the pass.
Termination (until): Ball is under control after receiving.
Condition (when): Teammate passes the ball back.
Sources: ['1', '2']

Task ID: 3
Objective (what): Pass the ball back to the teammate.
Control (how): Identify the teammate's position beyond opponent line and pass.
Termination (until): Ball reaches teammate in effective position.
Condition (when): Teammate breaches opponent line.
Sources: ['1', '2']


### Program Synthesis

In [14]:
# Edit the following to modify the task specification.

# self.what = what
# self.how = how
# self.until = until
# self.when = when

# self.sources = sources

In [15]:
from action import *

act = Act()
actions = Action.fromTask(tasks, actionsAPI)
act.do(actions)

for a in act.actions:
    print(a.id)
    print(str(a.task))
    print()

MoveTo
Task ID: 0
Objective (what): Get possession of the ball.
Control (how): Move towards the ball and control it.
Termination (until): Ball is under control.
Condition (when): 

Wait
Task ID: 0
Objective (what): Get possession of the ball.
Control (how): Move towards the ball and control it.
Termination (until): Ball is under control.
Condition (when): 

PassTo
Task ID: 1
Objective (what): Pass the ball to a teammate.
Control (how): Identify the teammate's position and pass.
Termination (until): Ball reaches the teammate.
Condition (when): 

Wait
Task ID: 2
Objective (what): Receive the ball from the teammate.
Control (how): Position yourself to receive the pass.
Termination (until): Ball is under control after receiving.
Condition (when): Teammate passes the ball back.

MoveTo
Task ID: 2
Objective (what): Receive the ball from the teammate.
Control (how): Position yourself to receive the pass.
Termination (until): Ball is under control after receiving.
Condition (when): Teammate pa

In [16]:
tasks_prompt = '\n\n'.join([f'[Action {idx} - {a.id}] {str(a.task)}' for idx, a in enumerate(act.actions)])

timing_instruction = """
For each of the actions ordered in chronological order (note that multiple actions might share task ID), you should find the frame that correspond to such action.
Such frame should be the moment where the action is being taken so the user can learn how to do it, for which the state of the scene (i.e. position and behaviour of the objects) is very important.
Your response should be formatted as a json

{
    'options': [
        'action': int,
        'frame': int
    ]
}
"""

entries = [
    Chat2.Entry(role='system', text=timing_instruction),
    Chat2.Entry(role='user', text=tasks_prompt)
] + [
    Chat2.Entry(role='user', text=f'Frame index: {idx}', im=get_im(im_dir)) for idx, im_dir in enumerate(frames)
]

chat = Chat2(client, model='o1')
response = json.loads(chat(entries, json=True))
print('Inference done.')

Inference done.


In [ ]:
from PIL import Image
from IPython.display import clear_output

times = [{} for a in act.actions]

def frameToTime(frame):
    return frame * 2.0

for i in response['options']:

    a = act.actions[i['action']]
    print(f'[{a.id}] {str(a.task)}')

    plt.imshow(Image.open(frames[i['frame']]))
    plt.axis('off')
    plt.show()

    time = frameToTime(i['frame'])

    mod = input('Type the index (0-indexed) of keyframe to modify it or return to continue.')

    if mod:
        time = float(mod)

    clear_output(wait=False)

    times[i['action']]['1'] = time

times

[{'1': 0.0},
 {'1': 4.0},
 {'1': 10.0},
 {'1': 14.0},
 {'1': 18.0},
 {'1': 20.0},
 {'1': 32.0}]

In [ ]:
times = [
    {
        '1': 5.0,
        '2': 3.0
    },
    {
        '1': 12.0,
        '2': 7.0
    },
    {
        '1': 20.0,
        '2': 14.0
    },
    {
        '1': 35.0,
        '2': 15.0
    },
    {
        '1': 35.0,
        '2': 15.0
    },
    {
        '1': 35.0,
        '2': 15.0
    },
    {
        '1': 35.0,
        '2': 15.0
    }
]

In [ ]:
for a, t in zip(act.actions, times):
    a.learn(map, t)

['1', '2']
[<scene.Scene object at 0x127beb690>, <scene.Scene object at 0x127beb750>] [5.0, 3.0]
{'logic': 'A', 'constraints': [{'id': 'A', 'api': 'HasBallPossession', 'params': {'ref': 'coach'}}], 'reasoning': 'The task requires the player to gain possession of the ball. The termination condition is when possession is achieved. Therefore, the constraint to check if the player has ball possession is used. Once the player has possession, the waiting action should be terminated.'}
reasoning The task requires the player to gain possession of the ball. The termination condition is when possession is achieved. Therefore, the constraint to check if the player has ball possession is used. Once the player has possession, the waiting action should be terminated.
['1', '2']
[<scene.Scene object at 0x127beb690>, <scene.Scene object at 0x127beb750>] [12.0, 7.0]
teammate
['1', '2']
[<scene.Scene object at 0x127beb690>, <scene.Scene object at 0x127beb750>] [20.0, 14.0]
{'logic': 'A AND B AND C', 'co

In [ ]:
actions_json = act.export()
print(actions_json)

{
    "actions": [
        {
            "id": "Idle",
            "args": {
                "precondition": "lambda_precondition"
            },
            "constraints": {
                "lambda_precondition": {
                    "logical": "A",
                    "identifiers": [
                        "A"
                    ],
                    "args": {
                        "A": {
                            "type": "HasBallPossession",
                            "args": {
                                "ref": "coach"
                            }
                        }
                    }
                }
            }
        },
        {
            "id": "PassTo",
            "args": {
                "obj": "teammate"
            }
        },
        {
            "id": "Idle",
            "args": {
                "precondition": "lambda_precondition"
            },
            "constraints": {
                "lambda_precondition": {
                    

### **Scenic** Translation

In [ ]:
def construct_coach_behavior(action_json):
    function_lines = ["behavior coachBehavior():"]
    function_lines.append("    scene = simulation()")

    for action in action_json["actions"]:
        if (action == None):
            continue
        action_id = action["id"]
        args = action.get("args", {})
        
        if action_id == "MoveTo":
            dest = args.get("dest", "")
            always = args.get("always", "")
            until = args.get("until", "")
            
            if dest and always and until:
                statement = f"    do {action_id}({dest}, {always}) until {until}"
            elif dest and until:
                statement = f"    do {action_id}({dest}) until {until}"
            elif dest:
                statement = f"    do {action_id}({dest})"
            else:
                statement = f"    do {action_id}()"
        
        elif action_id == "Idle":
            precondition = args.get("precondition", "")
            if precondition:
                statement = f"    do Idle() until {precondition}"
            else:
                statement = f"    do Idle()"

        elif action_id == "PassTo":
            obj = args.get("obj", "")
            if obj:
                statement = f"    do {action_id}({obj})"
            else:
                statement = f"    do {action_id}()"

        else:
            statement = f"    do {action_id}()"
        function_lines.append(statement)

    return "\n".join(function_lines)


In [ ]:
import re

def synthesize_conditionals(expression):
    expression = re.sub(r'\bIF\s+(.*?)\s+THEN\s+(.*?)\s+ELSE\s+(.*?)\b', r'(\2 if \1 else \3)', expression)
    expression = expression.replace("AND", "and").replace("OR", "or") 
    return expression

def get_args(action, constraint_name):
    args = action.get("constraints", {}).get(constraint_name, {}).get("args", {})
    formatted_args = []
    for arg_name, details in args.items():
        arg_type = details["type"]
        arg_values = ", ".join([f"'{key}': {repr(value)}" for key, value in details["args"].items()])
        formatted_args.append(f"{arg_name} = {arg_type}({{{arg_values}}})")
    return "\n".join(formatted_args)

def create_constraint_definitions(action_index, example):
    action = example["actions"][action_index]
    definitions = []
    for constraint_name in action.get("constraints", {}):
        constraint_def = get_args(action, constraint_name)
        definitions.append(constraint_def)
    return "\n".join(definitions)

def create_lambda_dest(action):
    constraints = action.get("constraints", {}).get("lambda_dest", {})
    lambda_def = "def λ_dest(scene, sample):\n"
    logical_expr = constraints.get("logical", "")
    if not logical_expr:
        return lambda_def + "    return None  # No logical expression provided\n"

    logical_expr = synthesize_conditionals(logical_expr)
    for constraint_name in constraints.get("args", []):
        verify_call = f"{constraint_name}(scene, sample)"
        logical_expr = logical_expr.replace(constraint_name, verify_call)
    
    lambda_def += f"    return {logical_expr}\n"
    return lambda_def

def create_lambda_termination(action):
    constraints = action.get("constraints", {}).get("lambda_termination", {})
    lambda_def = "def λ_termination(scene, sample):\n"
    logical_expr = constraints.get("logical", "")
    
    if not logical_expr:
        return lambda_def + "    return None  # No logical expression provided\n"

    logical_expr = synthesize_conditionals(logical_expr)
    for constraint_name in constraints.get("args", []):
        verify_call = f"{constraint_name}(scene, sample)"
        logical_expr = logical_expr.replace(constraint_name, verify_call)
    
    lambda_def += f"    return {logical_expr}\n"
    return lambda_def

def create_lambda_precondition(action):
    constraints = action.get("constraints", {}).get("lambda_precondition", {})
    lambda_def = "def λ_precondition(scene, sample):\n"
    logical_expr = constraints.get("logical", "")
    
    if not logical_expr:
        return lambda_def + "    return None  # No logical expression provided\n"

    logical_expr = synthesize_conditionals(logical_expr)
    for constraint_name in constraints.get("args", []):
        verify_call = f"{constraint_name}(scene, sample)"
        logical_expr = logical_expr.replace(constraint_name, verify_call)
    
    lambda_def += f"    return {logical_expr}\n"
    return lambda_def


In [ ]:
import json
actions_dict = json.loads(actions_json)
print(actions_dict)

{'actions': [{'id': 'Idle', 'args': {'precondition': 'lambda_precondition'}, 'constraints': {'lambda_precondition': {'logical': 'A', 'identifiers': ['A'], 'args': {'A': {'type': 'HasBallPossession', 'args': {'ref': 'coach'}}}}}}, {'id': 'PassTo', 'args': {'obj': 'teammate'}}, {'id': 'Idle', 'args': {'precondition': 'lambda_precondition'}, 'constraints': {'lambda_precondition': {'logical': 'A AND B AND C', 'identifiers': ['A', 'B', 'C'], 'args': {'A': {'type': 'HasBallPossession', 'args': {'ref': 'coach'}}, 'B': {'type': 'AheadOfLine', 'args': {'obj': 'teammate', 'height': {'avg': 6.05225, 'std': 0.0}}}, 'C': {'type': 'HasAngleOfPass', 'args': {'ref': 'teammate', 'radius': {'avg': 1.9283794442088684, 'std': 5.568196052607721e-05}}}}}}}, {'id': 'PassTo', 'args': {'obj': 'teammate'}}]}


In [ ]:
def generate_all_constraints_and_lambdas(example):
    for i, action in enumerate(example["actions"]):
        if (action is None):
            continue
        print(f"### Constraints and Lambda Functions for Action {i + 1}: {action['id']}")
        
        constraint_definitions = create_constraint_definitions(i, example)
        if constraint_definitions:
            print("Constraint Definitions:")
            print(constraint_definitions)
        else:
            print("No Constraint Definitions.")

        lambda_precondition_code = create_lambda_precondition(action)
        if lambda_precondition_code:
            print("\nλ_precondition Function:")
            print(lambda_precondition_code) 

        lambda_dest_code = create_lambda_dest(action)
        if lambda_dest_code:
            print("\nλ_dest Function:")
            print(lambda_dest_code)

        lambda_termination_code = create_lambda_termination(action)
        if lambda_termination_code:
            print("\nλ_termination Function:")
            print(lambda_termination_code)

        print("\n" + "=" * 40 + "\n")

generate_all_constraints_and_lambdas(actions_dict)

### Constraints and Lambda Functions for Action 1: Idle
Constraint Definitions:
A = HasBallPossession({'ref': 'coach'})

λ_precondition Function:
def λ_precondition(scene, sample):
    return A(scene, sample)


λ_dest Function:
def λ_dest(scene, sample):
    return None  # No logical expression provided


λ_termination Function:
def λ_termination(scene, sample):
    return None  # No logical expression provided



### Constraints and Lambda Functions for Action 2: PassTo
No Constraint Definitions.

λ_precondition Function:
def λ_precondition(scene, sample):
    return None  # No logical expression provided


λ_dest Function:
def λ_dest(scene, sample):
    return None  # No logical expression provided


λ_termination Function:
def λ_termination(scene, sample):
    return None  # No logical expression provided



### Constraints and Lambda Functions for Action 3: Idle
Constraint Definitions:
A = HasBallPossession({'ref': 'coach'})
B = AheadOfLine({'obj': 'teammate', 'height': {'avg': 6.0

In [ ]:
class InZone:
    def __init__(self, args):
        self.args = args

class HasAngle:
    def __init__(self, args):
        self.args = args

class IsVisible:
    def __init__(self, args):
        self.args = args

class DistanceLessThan:
    def __init__(self, args):
        self.args = args

class DistanceGreaterThan:
    def __init__(self, args):
        self.args = args


In [ ]:
behavior_code = construct_coach_behavior(actions_dict)
print(behavior_code)

behavior coachBehavior():
    scene = simulation()
    do Idle() until lambda_precondition
    do PassTo(teammate)
    do Idle() until lambda_precondition
    do PassTo(teammate)


In [ ]:
#Demo 1: (a) Once you are ready to realize that your teammate is ready to play [The expert referenced 'teammate' in the scene.] forward,then you want [The Coach passed the ball to teammate in the scene.] to pass a ball to him. (b) Once your teammate passes the ball back to you and they penetrate the [The expert referenced 'opponent_A' in the scene.] line behind [The expert referenced 'opponent_B' in the scene.] these players [The expert referenced 'opponent_C' in the scene.] [The expert referenced 'opponent_D' in the scene.],then you are ready to pass the ball back [The teammate passed the ball to Coach in the scene.] to him in order [The Coach passed the ball to teammate in the scene.] to penetrate this line of defense.

In [ ]:
# Segment 1

# precondition: 
#   has angle (beginning of segment)
#   ahead of line (end of segment)
# action: pass to teammate
# termination: none

# Segment 2

# precondition: 
#   has posession (beginning of segment)
#   player ahead of defenders (end of segment)
# action: pass to teammate
# termination: none